# <b> <font color='#A20025'> Appendix I &middot; Notebook 3 of 5: FlowMol3: Flow Matching for Real 3D Molecules</font> </b>

*This is a bonus notebook accompanying Appendix I of [Build AI Drug Discovery Pipelines](https://github.com/nrflynn2/ml-drug-discovery). It is self-contained and runnable end to end; read it side by side with the appendix. This notebook carries the flow-matching idea from a 2-D toy to de novo 3D drug-like molecules with a pretrained model, and revisits the step-count trade-off in molecular space.*

In Notebook 2 you watched flow matching redraw the two-moons target in a **handful of Euler steps**, where diffusion needed dozens. That was a 2-D cartoon. Now you carry the *identical* idea to a real drug-discovery task: **unconditional de-novo generation of 3D small molecules**; sampling brand-new drug-like molecules, atom coordinates and bond graph together, from noise.

You will not train anything. You will **load a pretrained checkpoint**; [FlowMol3](https://arxiv.org/abs/2508.12629) (Dunn & Koes, 2025), an open-source MIT-licensed model; and use it for inference only, on CPU. By the end you will have a set of sampled molecules written to an SDF file, their drug-likeness profiled, and the chapter's thesis re-tested on real molecules: **does flow matching still reach good samples in far fewer steps?**

> **What makes molecules harder than moons.** A 2-D point needs two continuous numbers. A molecule needs continuous **3D coordinates** *and* discrete **atom types** and **bond orders**. FlowMol3 is a *multi-modal* flow matcher: it flows the geometry and the chemistry jointly. We unpack that in §3.

## <b> <font color='#A20025'> Environment Setup</font> </b>

FlowMol3 depends on **DGL** (graph neural networks), **PyTorch**, **torch-scatter**, and **RDKit**. On a fresh machine or Colab the cell below installs a **CPU build** of the whole stack and the `flowmol` package itself, then downloads the pretrained checkpoint on first use.

> **Caution: The one install subtlety.** DGL loads a compiled `graphbolt` library whose filename is built from `torch.__version__`. A conda PyTorch build tags a local-version suffix (e.g. `2.1.2.post104`) that no shipped DGL library matches, so DGL fails to import. The fix is to install a **clean CPU wheel** (`torch==2.1.2` from the PyTorch CPU index, whose version string is exactly `2.1.2+cpu`) and the matching `torch-scatter` and `dgl` CPU wheels. The cell below pins those versions. On Colab this takes a few minutes the first time.

This notebook is **CPU-only**; no GPU required. Sample and step counts are kept modest so everything finishes on a laptop or a free Colab CPU runtime.

In [ ]:
# --- Colab-friendly setup: safe to re-run. CPU-only FlowMol3 stack. ---
import importlib, subprocess, sys, os

def _have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# 1) clean CPU PyTorch (version string '2.1.2+cpu' -> matches DGL's graphbolt lib)
if not _have('torch'):
    _pip('torch==2.1.2', '--index-url', 'https://download.pytorch.org/whl/cpu')
# 2) torch-scatter built against torch 2.1.2 cpu
if not _have('torch_scatter'):
    _pip('torch_scatter', '-f', 'https://data.pyg.org/whl/torch-2.1.2+cpu.html')
# 3) CPU DGL 2.1.x (ships libgraphbolt_pytorch_2.1.2.so, matching torch 2.1.2)
#    DGL's CPU wheels are served from its own index, keyed by torch minor version.
if not _have('dgl'):
    _pip('dgl==2.1.0', '-f', 'https://data.dgl.ai/wheels/torch-2.1/repo.html')
# 4) chemistry + model deps
for mod, pkg in [('rdkit','rdkit'), ('pytorch_lightning','pytorch-lightning==2.1.3'),
                 ('einops','einops'), ('pystow','pystow')]:
    if not _have(mod):
        _pip(pkg)
# 5) the flowmol package itself (MIT, github.com/Dunni3/FlowMol)
if not _have('flowmol'):
    _pip('git+https://github.com/Dunni3/FlowMol.git')

# CPU thread hygiene: the pip torch wheel bundles Intel OpenMP; disable thread
# affinity so it does not trip restricted (e.g. sandboxed/Colab) environments.
os.environ.setdefault('KMP_AFFINITY', 'disabled')
os.environ.setdefault('OMP_PROC_BIND', 'FALSE')
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')
print('Setup complete - CPU-only, no GPU required.')

In [ ]:
# --- Path setup: resolve data/figures relative to this notebook, so paths work on Colab and locally ---
from pathlib import Path
import os

# On Colab, clone the repo and move into this notebook's folder on first run.
if not Path("data").exists() and not Path("../data").exists():
    if os.path.exists("/content"):  # Colab
        os.system("git clone --depth 1 https://github.com/nrflynn2/ml-drug-discovery.git")
        os.chdir("ml-drug-discovery/Appendix_I_FLYNN_ML4DD/notebooks")

HERE = Path.cwd()
APPENDIX = HERE.parent if HERE.name == "notebooks" else HERE
DATA = APPENDIX / "data"
FIGURES = APPENDIX / "figures"
ARTIFACTS = APPENDIX / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("data dir:", DATA)

![Figure I.6]({{artifact:a5820035-3c43-423b-ba2d-90104b12ddca}})

**Figure I.6.** Validity versus the number of sampling steps, the toy thesis all over again in molecular space. The curve tracks the fraction of chemically valid molecules FlowMol3 produces as the integration budget grows; validity climbs steeply from a low value at 10 steps and then flattens once enough steps are spent, so beyond roughly 50 steps extra compute buys almost nothing. The plateau is the practical lesson: pick the smallest step budget on the flat part of the curve.

### <b> <font color='#A20025'> Import Packages</font> </b>

In [ ]:
import time, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, Draw, RDConfig, rdMolDescriptors
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

# standard RDKit synthetic-accessibility scorer (ships in RDKit Contrib)
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
import sascorer

import flowmol
torch.set_num_threads(4)
print('flowmol ready |', len(flowmol.pretrained_model_names), 'pretrained models available')

## <b> <font color='#A20025'> What flow matching does to a molecule</font> </b>

In Notebook 2 a sample was a point $x\in\mathbb{R}^2$, and flow matching learned one velocity field $v_\theta(x,t)$ that transported a Gaussian sample at $t{=}0$ to a data sample at $t{=}1$. A molecule is richer, so FlowMol3 flows **three modalities at once**:

- **atom coordinates** $x\in\mathbb{R}^{N\times 3}$; *continuous*. These flow exactly like the two-moons points: an **ODE** you integrate from noise to a clean 3D conformation.
- **atom types** and **bond orders**; *discrete* (carbon vs. nitrogen; single vs. double bond). You cannot draw a straight line from 'noise' to 'nitrogen'. FlowMol3 uses a **continuous-time Markov chain (CTMC)**: as $t$ advances, each atom/bond *jumps* between categories at a learned rate, arriving at a definite type by $t{=}1$.

So sampling one molecule means integrating a **coupled continuous-plus-discrete flow** from $t{=}0$ to $t{=}1$. The continuous part is the same ODE idea you already have; the discrete part is its categorical analogue. Both share **one clock** and the **same number of integration steps** `n_timesteps`; the knob at the center of this chapter.

> **Two views of the same step.** For coordinates, a step nudges every atom a little along its velocity: $x_{t+\Delta t} = x_t + v_\theta(x_t,t)\,\Delta t$. For a bond, the same step is a *chance to jump* to a new bond order, with probability set by the learned rate. Fewer steps = coarser moves in both.

This is the model's own framing: FlowMol3 is *"a multi-modal flow matching model"* that jointly samples molecular topology and 3D structure (Dunn & Koes, 2025).

## <b> <font color='#A20025'> Load the pretrained checkpoint</font> </b>

`flowmol.load_pretrained()` returns the default **`flowmol3`** model, trained on **GEOM-Drugs** (drug-like molecules with explicit hydrogens). The weights are **MIT-licensed** and download automatically on first call from the authors' server (`bits.csb.pitt.edu`) into `flowmol/trained_models/`. You load once and reuse.

We move the model to **CPU** and put it in `eval()` mode; this is inference only, no gradients, no training.

In [ ]:
t0 = time.time()
model = flowmol.load_pretrained('flowmol3').to('cpu').eval()
print(f'loaded flowmol3 in {time.time()-t0:.1f}s')
print('  parameterization :', model.parameterization, '  (ctmc = discrete CTMC + continuous ODE)')
print('  default timesteps:', model.default_n_timesteps)
n_params = sum(p.numel() for p in model.parameters())
print(f'  learnable params : {n_params/1e6:.1f}M')

## <b> <font color='#A20025'> Sample brand-new molecules</font> </b>

One call does it. `sample_random_sizes` first draws how many atoms each molecule should have (from the training-set size distribution), then integrates the coupled flow for `n_timesteps` steps. Pass **`device='cpu'`** (the default is a GPU). Each returned object carries a 3D RDKit molecule at `.rdkit_mol`.

We sample **12 molecules** at **100 steps**; modest, so it finishes on CPU in a few minutes. Scale `n_molecules` up on a GPU.

> These are genuinely novel structures drawn from the model, not retrieved from any database. Some will be chemically sensible, some won't; profiling that is §4.

If you are short on time or compute, the next cell can load the 12 molecules we already generated (set `USE_CACHED = True`) instead of sampling fresh.

In [ ]:
# FlowMol3 sampling. Set USE_CACHED = True to load the 12 molecules we generated for the
# appendix (instant, no model/GPU needed); set False to sample fresh (needs the model
# loaded above, about a minute on CPU).
USE_CACHED = True

N_MOLS   = 12
N_STEPS  = 100

if USE_CACHED:
    # Load the precomputed 3D molecules committed to the repo.
    from rdkit import Chem
    import os
    cached_path = str(DATA / "flowmol3_generated.sdf")
    if not os.path.exists(cached_path):
        cached_path = 'flowmol3_generated.sdf'   # fallback: local working copy
    raw_mols = [m for m in Chem.SDMolSupplier(cached_path, removeHs=False) if m is not None]
    print(f'loaded {len(raw_mols)} cached molecules from {cached_path}')
else:
    torch.manual_seed(2024)
    t0 = time.time()
    sampled = model.sample_random_sizes(n_molecules=N_MOLS, n_timesteps=N_STEPS, device='cpu')
    print(f'sampled {len(sampled)} molecules in {time.time()-t0:.1f}s')
    raw_mols = [s.rdkit_mol for s in sampled]

print('atom counts (with explicit H):', [m.GetNumAtoms() if m else 0 for m in raw_mols])

## <b> <font color='#A20025'> Are they real molecules? Validity, then SDF</font> </b>

A generated atom cloud is only a *molecule* if RDKit can **sanitize** it; valences satisfied, rings kekulizable, a chemically consistent graph. Following the FlowMol / MiDi convention, we take each molecule's **largest fragment** and call it valid if it sanitizes. FlowMol3's headline result is *"nearly 100% molecular validity for drug-like molecules with explicit hydrogens"* (Dunn & Koes, 2025); let's measure it on our small CPU batch.

Valid molecules are written to **`flowmol3_generated.sdf`** with their 3D coordinates, so you can open them in any molecular viewer or hand one to a docking tool.

In [ ]:
def largest_frag_sanitized(m):
    if m is None: return None, None
    try:
        frags = Chem.rdmolops.GetMolFrags(m, asMols=True, sanitizeFrags=False)
    except Exception:
        return None, None
    if not frags: return None, 0
    largest = max(frags, key=lambda f: f.GetNumAtoms())
    try:
        mm = Chem.Mol(largest); Chem.SanitizeMol(mm); return mm, len(frags)
    except Exception:
        return None, len(frags)

valid_mols, n_valid, n_conn = [], 0, 0
for raw in raw_mols:
    san, nf = largest_frag_sanitized(raw)
    if san is not None:
        n_valid += 1; valid_mols.append(san)
        if nf == 1: n_conn += 1

print(f'validity : {n_valid}/{len(raw_mols)} = {100*n_valid/len(raw_mols):.0f}%')
print(f'connected: {n_conn}/{len(raw_mols)} (single-fragment)')

# write valid molecules (3D) to SDF
w = Chem.SDWriter('flowmol3_generated.sdf')
for i, (raw, san) in enumerate(zip(raw_mols, [largest_frag_sanitized(m)[0] for m in raw_mols])):
    if san is None: continue
    # store a clean heavy-atom SMILES: RemoveHs alone leaves spurious chirality tags
    # on pseudo-centers (CH3, CF3) from the all-atom graph, so clear them too.
    heavy = Chem.RemoveHs(san)
    Chem.AssignStereochemistry(heavy, cleanIt=True, force=True)
    raw.SetProp('_Name', f'flowmol3_mol{i}')
    raw.SetProp('SMILES', Chem.MolToSmiles(heavy))
    w.write(raw)
w.close()
print('wrote valid 3D molecules to flowmol3_generated.sdf')

## <b> <font color='#A20025'> Drug-likeness: QED and synthetic accessibility</font> </b>

Validity says the graph is legal; it doesn't say the molecule is *good*. Two standard medicinal-chemistry lenses:

- **QED** (Quantitative Estimate of Drug-likeness), $0\to1$, higher is better; a blend of size, lipophilicity, polarity and other properties into one drug-likeness score.
- **SA score** (Synthetic Accessibility), $1\to10$, **lower is easier** to make; penalizes strained rings and exotic substructures.

We compute both for every valid molecule and plot their distributions. For orientation, approved oral drugs typically sit at QED ≈ 0.5-0.9 and SA ≈ 2-5.

> **Caution: Score on the heavy-atom graph.** FlowMol3 generates molecules with **explicit hydrogens**, but QED and SA are defined on the *heavy-atom* structure. Call `Chem.RemoveHs` before scoring; leaving the H atoms in silently inflates the SA score by several units (its fragment fingerprint counts the hydrogens). This is a common and easy-to-miss trap when profiling molecules from an all-atom generator.

In [ ]:
heavy = [Chem.RemoveHs(m) for m in valid_mols]   # score on heavy-atom graph
qed = np.array([QED.qed(m) for m in heavy])
sa  = np.array([sascorer.calculateScore(m) for m in heavy])
mw  = np.array([Descriptors.MolWt(m) for m in heavy])
logp= np.array([Descriptors.MolLogP(m) for m in heavy])
print(f'QED : mean {qed.mean():.2f}  range [{qed.min():.2f}, {qed.max():.2f}]')
print(f'SA  : mean {sa.mean():.2f}  range [{sa.min():.2f}, {sa.max():.2f}]')
print(f'MW  : mean {mw.mean():.0f} Da   logP mean {logp.mean():.2f}')

![Figure I.5]({{artifact:1a373726-fa27-49d8-970a-dbd443f21012}})

**Figure I.5.** The generated batch profiled against the property ranges typical of oral drugs. Each panel is a histogram of one molecular property (QED, synthetic accessibility, molecular weight, and cLogP) across the valid molecules FlowMol3 produced, with the shaded band marking the drug-like range and the dashed line the batch median. The medians land squarely in range (QED 0.74, SA 2.26, molecular weight 293, cLogP 3.77), showing the pretrained model generates chemically sensible, drug-like molecules rather than exotic outliers.

The figure below (built for the chapter with the shared figure style) shows the QED and SA distributions of the generated set. Most molecules land in drug-like territory on both axes; from a model that has never seen a docking score or a property label, only the shape of drug-like chemical space.

In [ ]:
# The chapter figure 'flowmol3_properties.png' is rendered from this same qed/sa array.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(qed, bins=8, range=(0,1), color='#A20025', alpha=0.85, edgecolor='white')
axes[0].set_xlabel('QED  (higher = more drug-like)'); axes[0].set_ylabel('count')
axes[0].set_title('Drug-likeness (QED)')
axes[1].hist(sa, bins=8, range=(1,10), color='#6d0019', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('SA score  (lower = easier to make)'); axes[1].set_ylabel('count')
axes[1].set_title('Synthetic accessibility')
fig.tight_layout(); plt.show()

## <b> <font color='#A20025'> A harder bar: is the geometry physically real?</font> </b>

Everything we have measured so far is a property of the molecular **graph**. Sanitization asks
whether valences add up, whether rings kekulize, whether the connectivity is chemically legal. QED and
SA score are computed from that same graph. A generator could pass all of them while placing its atoms
in positions no real molecule would ever adopt, because none of those checks ever looks at a coordinate.

That gap matters more for a 3D generator than for anything else in this appendix. FlowMol3's entire
premise is that it produces **conformations**, not just formulas; the ODE that carries atom positions
from noise to a molecule is the part we spent the whole notebook explaining. So the honest question is
not "did we get 100% validity" but "would a computational chemist accept these coordinates". We will
answer it with two independent tools, and the answer for this batch is genuinely good news; in Notebook 4
the same measurement on a pocket-conditioned generator tells a much less comfortable story.

**PoseBusters** is the bar the field settled on for exactly this question. Buttenschoen, Morris and Deane
built it after showing that deep-learning docking methods can report sub-2 Angstrom RMSD to a
crystallographic pose and still hand you a ligand with impossible bond lengths, flattened stereocentres,
or atoms sitting inside the protein. It runs a battery of independent physical tests, each a plain
pass or fail, in three families: **chemistry** (sanitization, InChI convertibility, connectivity, no
radicals), **intramolecular geometry** (bond lengths and angles against reference distributions, internal
steric clash, aromatic ring flatness, non-aromatic ring non-flatness, double-bond flatness), and an
**internal energy ratio** that compares the pose's UFF energy to a relaxed conformer ensemble. FlowMol3's
molecules float in empty space with no protein frame, so we run the `mol` configuration, which is
precisely the subset of checks that need no receptor: eleven intramolecular tests and nothing else.

In [ ]:
# PoseBusters intramolecular validity on the FlowMol3 batch.
# USE_CACHED = True reads the result we committed to the repo (instant). Set it to False to
# recompute from the molecules in memory: about 23 s for 12 molecules on a laptop CPU, and it
# needs `pip install posebusters` (not part of the FlowMol stack installed above).
import io
import pandas as pd

USE_CACHED = True

# Every metrics CSV in this appendix carries a provenance header as leading '#' lines. Strip
# ONLY those leading lines by hand. Do NOT pass comment='#' to read_csv: a '#' inside a SMILES
# string is a triple bond (C#N is a nitrile), and pandas would truncate that field mid-value
# and silently leave NaN behind. Two of the 20 DiffSBDD SMILES in Notebook 4 are nitriles, so
# this is a real bug, not a hypothetical one.
def read_metrics_csv(path):
    lines = open(path).read().splitlines()
    i = 0
    while i < len(lines) and lines[i].lstrip().startswith("#"):
        i += 1
    return pd.read_csv(io.StringIO("\n".join(lines[i:])))


PB_CSV = DATA / "posebusters_flowmol3.csv"
if not PB_CSV.exists():
    PB_CSV = Path("posebusters_flowmol3.csv")      # fallback: local working copy

# The eleven intramolecular checks the 'mol' configuration reports, in PoseBusters' own order.
INTRA_CHECKS = ['sanitization', 'inchi_convertible', 'all_atoms_connected', 'no_radicals',
                'bond_lengths', 'bond_angles', 'internal_steric_clash', 'aromatic_ring_flatness',
                'non-aromatic_ring_non-flatness', 'double_bond_flatness', 'internal_energy']

if USE_CACHED:
    pb = read_metrics_csv(PB_CSV)
else:
    from posebusters import PoseBusters
    # valid_mols are the sanitized largest fragments with explicit hydrogens and their 3D
    # coordinates intact, which is what the geometry checks need. config='mol' is the subset
    # of checks that require no receptor; see Notebook 4 for the protein-aware configuration.
    pb_raw = PoseBusters(config='mol').bust(mol_pred=valid_mols, full_report=False)
    pb = pb_raw.reset_index()[INTRA_CHECKS]

# A check can come back NaN when it cannot be evaluated at all (UFF has no parameters for an
# exotic atom type, so the energy ratio is undefined). Report both readings rather than picking
# one: 'evaluated' skips NaN, 'strict' counts an unevaluable check as a failure.
checks = pb[INTRA_CHECKS].astype('boolean')
n_pass_evaluated = int(checks.apply(lambda row: row.dropna().all(), axis=1).sum())
n_pass_strict = int(checks.fillna(False).all(axis=1).sum())
n_mols = len(pb)

print(f'PoseBusters-valid (all evaluated checks pass): {n_pass_evaluated}/{n_mols}'
      f'  = {100*n_pass_evaluated/n_mols:.0f}%')
print(f'PoseBusters-valid (unevaluable counted as failure): {n_pass_strict}/{n_mols}')
print(f'checks that could not be evaluated: {int(checks.isna().sum().sum())}\n')
print('per-check pass rate:')
for check in INTRA_CHECKS:
    rate = float(checks[check].mean())
    print(f'  {check:<32s} {100*rate:5.1f}%')

Twelve out of twelve, on every one of the eleven checks, and nothing came back unevaluable. Bond
lengths and angles sit inside the reference distributions, no ring is flattened or puckered where it
should not be, no pair of atoms is jammed into another, and the internal energy ratio is acceptable for
all twelve. That is the result we hoped for and it is worth stating plainly, because the same code in
Notebook 4 returns 15 out of 20. FlowMol3's coordinates are not merely legal as a graph; they are
geometry a chemist would recognize.

One caveat on how far this generalizes. Twelve molecules is a small batch, sampled on a CPU for a
notebook, so treat this as "our batch holds up under scrutiny" rather than a benchmark of the model.
The published FlowMol3 evaluation runs thousands of samples against the GEOM-Drugs test distribution.
What we can say is that the specific molecules in this notebook, including the one we hand to a docking
connector in section I.7, are physically defensible.

In [ ]:
# UFF strain energy: the measurement that has no pass/fail line and is the sharpest signal here.
# PoseCheck defines strain as the pose's UFF energy minus the energy of a relaxed conformer
# ensemble for the same molecule, in kcal/mol. Low means the conformation is close to relaxed;
# high means the generator bent the molecule into something thermally implausible.
# Fresh compute needs `pip install posecheck` and runs about 2.4 s per molecule.
USE_CACHED = True

STRAIN_CSV = DATA / "posecheck_strain_flowmol3.csv"
if not STRAIN_CSV.exists():
    STRAIN_CSV = Path("posecheck_strain_flowmol3.csv")
# The caffeine yardstick lives in the DiffSBDD file, because that is where the co-crystal
# reference pose was measured. Strain needs no protein, so the number transfers unchanged.
REF_CSV = DATA / "posecheck_diffsbdd.csv"
if not REF_CSV.exists():
    REF_CSV = Path("posecheck_diffsbdd.csv")

if USE_CACHED:
    strain = read_metrics_csv(STRAIN_CSV)['strain_energy_kcal_mol']
else:
    from posecheck import PoseCheck
    pc = PoseCheck()
    pc.load_ligands_from_mols(valid_mols)        # no protein: strain is intramolecular
    # Expect your numbers to land near ours rather than on them. PoseCheck builds the relaxed
    # reference ensemble with AllChem.EmbedMolecule and passes no random seed, so the ensemble
    # minimum, and therefore the strain, moves a little from run to run. Recomputing this batch
    # gave a median of 8.33 against the 8.18 in the cached file, about 2% apart.
    strain = pd.Series(pc.calculate_strain_energy())

ref = read_metrics_csv(REF_CSV)
caffeine_strain = float(ref.loc[ref['is_reference'].astype(bool), 'strain_energy_kcal_mol'].iloc[0])

print(f'FlowMol3 strain (kcal/mol): median {strain.median():.2f}'
      f'   IQR [{strain.quantile(0.25):.2f}, {strain.quantile(0.75):.2f}]'
      f'   max {strain.max():.2f}')
print(f'caffeine co-crystal pose:   {caffeine_strain:.3f}   (the 3RFM reference ligand, measured the same way)')
print(f'fold over the co-crystal:   {strain.median()/caffeine_strain:.1f}x')

A median of 8.18 kcal/mol against the caffeine co-crystal pose's 0.39. Read the ratio honestly: it is
21x the reference, which is not nothing. A crystallographic pose has been refined against experimental
density and sits essentially at the bottom of its energy well, so it is a demanding yardstick, and a
handful of kcal/mol of strain is within what a real ligand pays to bind. What matters is the order of
magnitude. FlowMol3's molecules are strained the way a slightly awkward conformer is strained, not the
way a broken molecule is; hold that number in mind, because the pocket-conditioned generator in
Notebook 4 has a median of 134.5.

In [ ]:
# REOS (Rapid Elimination Of Swill): substructure alerts for functional groups that make a
# molecule a poor screening candidate. This is FlowMol's own quality signal, and it is a
# different question from geometry: is the CHEMISTRY something you would put in a plate?
# The rule set is Glaxo, the useful_rdkit_utils default, 55 rules. A molecule is flagged for a
# rule when its substructure-match count exceeds that rule's allowed maximum.
REOS_CSV = DATA / "reos_flowmol3.csv"
if not REOS_CSV.exists():
    REOS_CSV = Path("reos_flowmol3.csv")

reos = read_metrics_csv(REOS_CSV)
n_rules = len(reos)
n_scored = int(reos['n_mols'].iloc[0])

print(f'rule set: Glaxo, {n_rules} rules  |  molecules scored: {n_scored}')
print(f'total FlowMol3 flags across all rules: {int(reos["flag_count"].sum())}')
n_flagged_rules = int((reos["flag_count"] > 0).sum())
print(f'rules that fired at least once:        {n_flagged_rules}\n')

flagged = reos[reos['flag_count'] > 0][['flag_name', 'flag_count', 'flag_rate']]
if flagged.empty:
    print('no rule fired on any FlowMol3 molecule.')
else:
    print(flagged.to_string(index=False))

# The same file carries the DiffSBDD batch under the same 55 rules, for contrast.
d_flagged = reos[reos['flag_count_diffsbdd'] > 0][['flag_name', 'flag_count_diffsbdd', 'flag_rate_diffsbdd']]
print(f'\nfor contrast, DiffSBDD (Notebook 4, {int(reos["n_mols_diffsbdd"].iloc[0])} ligands):')
print(d_flagged.to_string(index=False))

Zero flags out of 55 rules across all 12 molecules. No epoxide, no acid halide or anhydride, no
peroxide, no quinone, no long aliphatic methylene chain; nothing on the Glaxo list of substructures a
screening group would rather not receive. The DiffSBDD comparison in the same file shows what a non-zero
result looks like: three of its 20 ligands trip a rule, two on the epoxide, thioepoxide and aziridine
rule and one on an aliphatic methylene chain of seven or more carbons. Neither model was trained with a
structural-alert filter, and neither result is a large enough sample to rank the two generators on; three
flags out of 20 molecules could easily be one or five in another batch. It is simply the last box to tick
before we call this batch clean, and a useful reminder that structural alerts are a different axis from
geometry: a molecule can be perfectly relaxed and still carry a substructure nobody wants in a plate.

![Figure I.7b]({{artifact:56ef2f07-1a33-4ea7-8e87-a29b82d2216b}})

**Figure I.7b.** Physical plausibility for both generators in this appendix, on the three axes that
graph validity cannot see. Panel (a) gives the eleven intramolecular PoseBusters checks as the percentage
of molecules passing, FlowMol3 in slate and DiffSBDD in crimson: FlowMol3 clears every check at 100%,
while DiffSBDD drops to 90% on bond lengths, 85% on bond angles and 95% on non-aromatic ring
non-flatness, and its energy-ratio check could only be evaluated for 18 of 20 molecules. Panel (b) shows
the nine ligand-to-protein checks, which exist only for DiffSBDD because FlowMol3 samples into empty
space with no receptor; every one of them passes for all 20 ligands, so the pocket-conditioned model
places its molecules correctly with respect to the protein. Panel (c) plots UFF strain energy on a log
scale, one point per molecule, with the caffeine co-crystal pose as the dashed reference at 0.39
kcal/mol: FlowMol3's median of 8 kcal/mol sits an order of magnitude above the crystal pose, and
DiffSBDD's median of 134 sits an order of magnitude above that, with its worst ligand at 2601. Together
the panels locate DiffSBDD's problem precisely: it is intramolecular geometry, not placement, and it is
invisible to both the 100% validity figure and to PoseBusters' pass or fail battery.

## <b> <font color='#A20025'> Novel compared to what? Naming the metric</font> </b>

Two numbers get quoted about generative chemistry more than any others, and both are meaningless
until you say what they were measured against. "Novelty" is a Tanimoto similarity, and Tanimoto is
computed between two **fingerprints**, so the number depends entirely on which molecules you compare to
and which fingerprint you use. "Diversity" has the same problem twice over. This section makes both
concrete for FlowMol3's batch, so that every figure we print can be recomputed by a reader.

Two definitions first. A **fingerprint** is a fixed-length bit vector summarizing a molecule's
substructures, and **Tanimoto similarity** is the size of the intersection of two such bit sets over the
size of their union, running from 0 (no shared bits) to 1 (identical bits). We use two fingerprints
throughout, and they answer subtly different questions. `Chem.RDKFingerprint` at RDKit's defaults is
**path-based**: it hashes linear atom paths of length 1 to 7 into 2048 bits, so radius is not one of its
parameters. `Morgan r=2/2048` is **circular** (the ECFP4 family): it hashes the environment within two
bonds of each atom into 2048 bits. Neither is more correct, but they are not interchangeable, and a
similarity number without its fingerprint is not reproducible.

The question that actually matters for a generative model is **not** how different its output is from
one reference ligand. It is whether the model is quietly reciting its training set. Those are different
measurements, and the gap between them is large. We compute both here: the maximum Tanimoto from each
generated molecule to a sample of the training corpus (its nearest training neighbour), and the
similarity to caffeine, the reference ligand from the pocket in Notebook 4, under both fingerprints.

In [ ]:
# Novelty measured two ways, under two fingerprints, for the FlowMol3 batch.
# USE_CACHED = True reads the per-ligand table we committed. Set it to False to recompute
# against the sampled GEOM-Drugs corpus that ships alongside it (about 80 s: 12 generated
# molecules against 24,197 corpus molecules, twice).
USE_CACHED = True

NOV_CSV = DATA / "novelty_flowmol3.csv"
if not NOV_CSV.exists():
    NOV_CSV = Path("novelty_flowmol3.csv")
CORPUS_SMI = DATA / "corpus_sample_geom_drugs.smi"
if not CORPUS_SMI.exists():
    CORPUS_SMI = Path("corpus_sample_geom_drugs.smi")

from rdkit.Chem import DataStructs, rdFingerprintGenerator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def both_fingerprints(mol):
    """Path-based RDKit fingerprint at defaults, and circular Morgan r=2, both 2048 bits."""
    return Chem.RDKFingerprint(mol), morgan_gen.GetFingerprint(mol)


if USE_CACHED:
    nov = read_metrics_csv(NOV_CSV)
else:
    # The corpus file is tab-separated SMILES with a leading '#' provenance header.
    lines = open(CORPUS_SMI).read().splitlines()
    i = 0
    while i < len(lines) and lines[i].lstrip().startswith("#"):
        i += 1
    corpus_smiles = [ln.split('\t')[0] for ln in lines[i:] if ln.strip()]
    corpus_fps = [both_fingerprints(m) for m in
                  (Chem.MolFromSmiles(s) for s in corpus_smiles) if m is not None]
    corpus_rdkit = [a for a, _ in corpus_fps]
    corpus_morgan = [b for _, b in corpus_fps]
    print(f'corpus: {len(corpus_rdkit)} unique GEOM-Drugs molecules')

    caffeine = Chem.MolFromSmiles('Cn1c(=O)c2c(ncn2C)n(C)c1=O')   # the 3RFM reference ligand
    caff_rdkit, caff_morgan = both_fingerprints(caffeine)

    rows = []
    for mol in [Chem.RemoveHs(m) for m in valid_mols]:
        fp_rdkit, fp_morgan = both_fingerprints(mol)
        rows.append({
            'smiles': Chem.MolToSmiles(mol),
            'max_tanimoto_vs_train_rdkit': max(DataStructs.BulkTanimotoSimilarity(fp_rdkit, corpus_rdkit)),
            'max_tanimoto_vs_train_morgan': max(DataStructs.BulkTanimotoSimilarity(fp_morgan, corpus_morgan)),
            'tanimoto_vs_caffeine_rdkit': DataStructs.TanimotoSimilarity(fp_rdkit, caff_rdkit),
            'tanimoto_vs_caffeine_morgan': DataStructs.TanimotoSimilarity(fp_morgan, caff_morgan)})
    nov = pd.DataFrame(rows)

print(f'{len(nov)} molecules  |  median Tanimoto, higher = more similar\n')
print(f'{"":<34s}{"RDKit path-based":>18s}{"Morgan r=2":>14s}')
for label, rd, mg in [
        ('nearest GEOM-Drugs neighbour', 'max_tanimoto_vs_train_rdkit', 'max_tanimoto_vs_train_morgan'),
        ('caffeine (the reference ligand)', 'tanimoto_vs_caffeine_rdkit', 'tanimoto_vs_caffeine_morgan')]:
    print(f'{label:<34s}{nov[rd].median():>18.4f}{nov[mg].median():>14.4f}')

for tag, rd, mg in [('training corpus / caffeine ratio',
                     'max_tanimoto_vs_train_rdkit', 'max_tanimoto_vs_train_morgan')]:
    r_rd = nov[rd].median() / nov['tanimoto_vs_caffeine_rdkit'].median()
    r_mg = nov[mg].median() / nov['tanimoto_vs_caffeine_morgan'].median()
    print(f'\n{tag:<34s}{r_rd:>17.2f}x{r_mg:>13.2f}x')

The two measurements disagree by more than a factor of two, in the direction that matters. Under the
path-based fingerprint, FlowMol3's molecules sit at median 0.2493 Tanimoto to caffeine, which sounds
reassuringly novel, and at median 0.5839 to their nearest neighbour in a sample of GEOM-Drugs, the
corpus the model was trained on. That is 2.34x higher. Under Morgan r=2 the same comparison is 0.1019
against caffeine and 0.3951 against the training sample, a ratio of 3.88x. Dissimilarity to one
reference ligand is simply not evidence about memorization, and only the second column speaks to it.

Two honest limits on the training-corpus number, both of which push the same way.

First, provenance. FlowMol3 was trained on GEOM-Drugs, and the corpus we compare against is a sample of
the published GEOM-Drugs training SMILES list taken from a public mirror of the standard 5-conformer
preprocessing (24,197 unique molecules drawn from 60 random byte offsets, seed 4242, CC-BY-4.0). It is
not guaranteed byte-identical to the exact split the released checkpoint saw, because that split is not
published as a SMILES list. Notebook 4 has a tighter version of this hedge: for DiffSBDD, 1,749 of its
sampled corpus molecules are independently confirmed present in the model's own published train split,
and restricting to them changes the answer very little.

Second, and more important, **these are lower bounds**, and the next cell shows why.

In [ ]:
# How much does the answer depend on how much corpus we searched? A nearest-neighbour
# similarity can only RISE as the corpus grows, so this curve tells us whether the number above
# is near its limit or simply the closest match we happened to look at. Nested prefixes of one
# seeded shuffle, so each corpus contains the one before it and the comparison is apples to
# apples. Runtime is dominated by the largest corpus: roughly 80 s in total.
import numpy as np

lines = open(CORPUS_SMI).read().splitlines()
i = 0
while i < len(lines) and lines[i].lstrip().startswith("#"):
    i += 1
corpus_smiles = [ln.split('\t')[0] for ln in lines[i:] if ln.strip()]
corpus_rdkit = [Chem.RDKFingerprint(m) for m in
                (Chem.MolFromSmiles(s) for s in corpus_smiles) if m is not None]
gen_rdkit = [Chem.RDKFingerprint(m) for m in
             (Chem.MolFromSmiles(s) for s in nov['smiles']) if m is not None]

order = np.random.default_rng(4242).permutation(len(corpus_rdkit))
print(f'{"corpus size":>12s}{"median-of-max Tanimoto":>26s}')
for divisor in (8, 4, 2, 1):
    n = len(order) // divisor
    subset = [corpus_rdkit[k] for k in order[:n]]
    nearest = [max(DataStructs.BulkTanimotoSimilarity(fp, subset)) for fp in gen_rdkit]
    print(f'{n:>12d}{float(np.median(nearest)):>26.4f}')

The curve climbs the whole way and never flattens: 0.5266 at 3,024 corpus molecules, 0.5457 at 6,049,
0.5749 at 12,098, and 0.5839 at the full 24,197-molecule sample. A nearest-neighbour search can only find
closer matches as you give it more molecules to search, never further ones, and we read only about 11% of
the published train SMILES file, so the real training set is roughly ten times what we searched here. The
true similarity between these molecules and FlowMol3's training distribution is therefore **higher** than
0.5839. We report the number we can compute and label it a floor, which is the honest form of the claim
and also the stronger one.

In [ ]:
# Diversity and uniqueness, with the definitions written into the code.
#   diversity  = 1 - mean pairwise Tanimoto over the valid molecules
#   uniqueness = distinct canonical SMILES / number of valid molecules
# Both are computed under both fingerprints, because the fingerprint changes the answer.
from itertools import combinations

heavy_mols = [Chem.RemoveHs(m) for m in valid_mols]


def diversity_and_uniqueness(mols, fingerprint):
    fps = [fingerprint(m) for m in mols]
    pairs = list(combinations(range(len(fps)), 2))
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j]) for i, j in pairs]
    # np.mean rather than sum()/len(): pairwise summation shifts the last digits of the float.
    diversity = 1 - float(np.mean(sims))
    n_distinct = len({Chem.MolToSmiles(m) for m in mols})
    return diversity, len(pairs), n_distinct


for name, fingerprint in [('RDKit path-based (fpSize=2048, minPath=1, maxPath=7, nBitsPerHash=2)',
                           Chem.RDKFingerprint),
                          ('Morgan circular (radius=2, fpSize=2048)',
                           morgan_gen.GetFingerprint)]:
    div, n_pairs, n_distinct = diversity_and_uniqueness(heavy_mols, fingerprint)
    print(f'{name}\n  diversity  = {div:.4f}   over {n_pairs} pairs of {len(heavy_mols)} molecules')
    print(f'  uniqueness = {n_distinct/len(heavy_mols):.2f}   ({n_distinct} distinct canonical SMILES)')

Diversity 0.7894 under the path-based fingerprint and 0.8695 under Morgan, from the same twelve
molecules. The spread of 0.08 is the whole point: a diversity figure quoted without its fingerprint,
its bit length and its formula is not a reproducible measurement, and the higher-sounding number is not
the better model. Uniqueness is 1.00 either way, all twelve canonical SMILES distinct, so the model is
not drawing the same molecule twice. When you report these numbers, report them the way this cell prints
them: metric, fingerprint, parameters, and the formula that combines them.

## <b> <font color='#A20025'> The thesis, on real molecules: quality vs. number of steps</font> </b>

Here is the chapter's central claim, back for its second test. On the two-moons toy, flow matching reached a clean target in a **handful of Euler steps** while diffusion needed dozens. Does the same hold when the samples are real 3D drug molecules?

We fix the molecule sizes **and the starting noise**, then sample the *same* batch at increasing step budgets; `n_timesteps` ∈ {10, 25, 50, 100, 250}; and track validity and mean QED. Because only the step count changes, any improvement is purely the integrator taking finer steps.

This is the exact experiment from Notebook 2, lifted from $\mathbb{R}^2$ to molecular space. Keep the counts modest so it finishes on CPU.

In [ ]:
# Reproduces the study saved as 'flowmol3_stepcount.png' / 'flowmol3_metrics.csv'.
# (Run in the companion script for the chapter; a few minutes on CPU.)
step_grid = [10, 25, 50, 100, 250]
torch.manual_seed(2024)
n_atoms = model.sample_n_atoms(N_MOLS).to('cpu')   # fix sizes once
results = []
for T in step_grid:
    torch.manual_seed(2024)                         # identical prior noise each T
    mols = model.sample(n_atoms, n_timesteps=T, device='cpu')
    sans = [largest_frag_sanitized(m.rdkit_mol)[0] for m in mols]
    nv = sum(s is not None for s in sans)
    q  = np.mean([QED.qed(s) for s in sans if s is not None])
    results.append({'n_timesteps': T, 'validity_pct': 100*nv/len(mols), 'mean_qed': q})
    print(f'T={T:3d}  validity {100*nv/len(mols):3.0f}%  mean QED {q:.2f}')

Running the study on this CPU batch gives:

| n_timesteps | validity % | connected % | mean QED | mean SA |
|---:|---:|---:|---:|---:|
| 10 | 42 | 75 | 0.52 | 5.23 |
| 25 | 83 | 75 | 0.62 | 3.50 |
| 50 | 100 | 100 | 0.73 | 2.79 |
| 100 | 100 | 100 | 0.72 | 2.43 |
| 250 | 100 | 100 | 0.67 | 2.32 |


**Read the thesis off this table.** Validity climbs **42% → 83% → 100%** as you go from 10 to 50 steps, then **plateaus**; 50, 100, and 250 steps all give 100% valid, fully-connected molecules. Synthetic accessibility improves the same way (SA falls from ~5.2 to ~2.3, *lower is easier*), and QED sits in the drug-like 0.6-0.7 band once you clear ~50 steps. This is exactly the pattern you saw on two-moons in Notebook 2: **flow matching reaches its target in a modest number of steps, and spending more barely moves the needle.** On a 2-D toy that meant clean moons at 4 steps; on real molecules it means 100% validity at ~50. The chapter figure `flowmol3_stepcount.png` plots this curve.

> **Why fewer steps matters here.** Each step is a full pass of a graph neural network over every atom and bond. Halving the steps roughly halves the wall-clock cost of generating a library. Flow matching's straight, few-step paths are what make generating thousands of candidate molecules practical.

## <b> <font color='#A20025'> Pick one molecule to carry forward</font> </b>

You now have a set of valid, drug-like 3D molecules. To close the chapter's loop, we pull out **one clean candidate**; the highest-QED molecule in the batch; and print its canonical SMILES. In §I.7 you hand exactly this string to a structure-prediction / docking connector (Boltz) to see how a *generated* ligand scores against a real protein target; the final step from noise to a testable hypothesis.

In [ ]:
best = max(valid_mols, key=lambda m: QED.qed(Chem.RemoveHs(m)))
best_heavy = Chem.RemoveHs(best)
Chem.AssignStereochemistry(best_heavy, cleanIt=True, force=True)  # drop pseudo-center stereo
handoff_smiles = Chem.MolToSmiles(best_heavy)
print('handoff SMILES:', handoff_smiles)
print(f'  QED {QED.qed(best_heavy):.3f} | SA {sascorer.calculateScore(best_heavy):.2f} '
      f'| MW {Descriptors.MolWt(best_heavy):.0f} | rings {rdMolDescriptors.CalcNumRings(best_heavy)}')
# -> feeds Section 10.7 (Boltz structure-prediction / docking connector)

For this run the pick was **`Cc1ccc(NC(=O)c2ccc(OC(F)(F)F)cc2)cc1`**; a trifluoromethoxy diaryl amide (QED ≈ 0.92, SA ≈ 1.6), a realistic, easily-synthesized scaffold. Your run will differ; the recipe is what matters.

## <b> <font color='#A20025'> Summary</font> </b>

You took the flow-matching idea from a 2-D toy to a real drug-discovery task, using a pretrained model for inference only:

- FlowMol3 is a **multi-modal flow matcher**: continuous **atom coordinates** flow by an **ODE** (the two-moons idea), while discrete **atom types and bond orders** flow by a **CTMC** that jumps between categories; one shared clock, one `n_timesteps` knob.
- `flowmol.load_pretrained('flowmol3')` gave you an MIT-licensed, GEOM-Drugs-trained model that **downloads its own weights** and runs on **CPU**.
- A modest CPU batch produced **valid, drug-like 3D molecules**; written to SDF with coordinates, profiled by **QED** and **SA**; with no training and no property labels.
- The **step-count study revisited the chapter thesis on real molecules**: FlowMol3 reaches near-plateau validity and drug-likeness at **modest step counts**, echoing the few-steps win you first saw on two-moons.

**Next:** you have a clean, valid molecule in hand. In §I.7 you hand it to a structure-prediction/docking connector (Boltz) to see how a generated ligand fares against a real target; closing the loop from noise to a candidate.

### <b> <font color='#A20025'> References</font> </b>

1. Dunn, I. & Koes, D. R. (2025), *FlowMol3: Flow Matching for 3D De Novo Small-Molecule Generation*. arXiv:2508.12629. Code (MIT): https://github.com/Dunni3/FlowMol
2. Dunn, I. & Koes, D. R. (2024), *Mixed Continuous and Categorical Flow Matching for 3D De Novo Molecule Generation* (FlowMol1). arXiv:2404.19739.
3. Dunn, I. & Koes, D. R. (2024), *Exploring Discrete Flow Matching for 3D De Novo Molecule Generation* (FlowMol2 / CTMC). arXiv:2411.16644.
4. Lipman, Chen, Ben-Hamu, Nickel & Le (2023), *Flow Matching for Generative Modeling*, ICLR. arXiv:2210.02747.
5. Vignac et al. (2023), *MiDi: Mixed Graph and 3D Denoising Diffusion for Molecule Generation*, ECML-PKDD. *(validity convention & GEOM-Drugs splits)*
6. Bickerton et al. (2012), *Quantifying the chemical beauty of drugs*, Nat. Chem. *(QED)*. Ertl & Schuffenhauer (2009), *J. Cheminform.* *(SA score)*.